# [2026-08-01-03] Clinical Trials by Recruitment Status

Self-contained notebook that compares the number of **interventional studies registered on
ClinicalTrials.gov** for ME/CFS and CrunchME's adult/all-ages comparator diseases.

Trials are shown as a stacked horizontal bar chart by their **overall recruitment status**.
`Suspended`, `Terminated` and `Withdrawn` are combined as `Stopped early`; any residual
status is retained as `Unknown / other`, so every interventional study is counted once.
Run all cells top to bottom; the final cells render the CrunchME-style chart and export the
source data and image.

**Default scope:** ME/CFS plus the adult core comparator groups from CrunchME's comparator
framework. Optional adult and Long COVID comparator groups can be enabled in section 2.2.


### 📦 1 | Install & Import


#### 1.1 | Install packages


In [ ]:
# Colab already includes most of these packages. Install only anything that is missing.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "requests": "requests",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "plotnine": "plotnine",
    "mizani": "mizani",
}

for package, module in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )


#### 1.2 | Import packages


In [ ]:
from collections import OrderedDict
from datetime import datetime, timezone
import hashlib
import json
import time

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import pandas as pd
import requests
from mizani.formatters import comma_format
from plotnine import (
    aes,
    coord_flip,
    element_blank,
    element_line,
    element_rect,
    element_text,
    geom_col,
    geom_text,
    ggplot,
    guide_legend,
    guides,
    labs,
    scale_fill_manual,
    scale_x_discrete,
    scale_y_continuous,
    theme,
    theme_classic,
)


#### 1.3 | Load CrunchME brand font (Brutel)

Searches common Colab/Drive locations for `Brutel*.otf` or `Brutel*.ttf`. If none are
found, Colab offers an upload prompt. The chart still renders with a sans-serif fallback.


In [ ]:
BRAND_FONT = "Brutel"

FONT_SEARCH_DIRS = [
    ".",
    "/content",
    "/content/fonts",
    "/content/drive/MyDrive/CrunchME/Fonts",
    "/content/drive/MyDrive/CrunchME Visual Branding/Font/brutel",
]

def _find_brutel(dirs):
    import glob
    import os

    hits = []
    for directory in dirs:
        for extension in ("otf", "ttf"):
            hits += glob.glob(
                os.path.join(directory, "**", f"Brutel*.{extension}"),
                recursive=True,
            )
    return sorted(set(hits))

font_files = _find_brutel(FONT_SEARCH_DIRS)

if not font_files:
    try:
        from google.colab import files

        print(
            "No Brutel files found. Upload Brutel-Regular and Brutel-Bold "
            "(other weights are optional):"
        )
        files.upload()
        font_files = _find_brutel(["."])
    except Exception as exc:
        print("Automatic font upload unavailable:", exc)

for font_path in font_files:
    try:
        fm.fontManager.addfont(font_path)
    except Exception as exc:
        print("Could not register", font_path, "-", exc)

available_fonts = {font.name for font in fm.fontManager.ttflist}
if BRAND_FONT in available_fonts:
    plt.rcParams["font.family"] = BRAND_FONT
    print(
        f"✓ Brand font '{BRAND_FONT}' active "
        f"({len(font_files)} file(s) newly registered)."
    )
else:
    BRAND_FONT = "DejaVu Sans"
    plt.rcParams["font.family"] = BRAND_FONT
    print("⚠ Brutel not found - using DejaVu Sans.")


### 🔧 2 | Parameters


#### 2.1 | Comparator sets


In [ ]:
# Comparator groups transcribed from CrunchME's 1 August 2026 framework.
# The two ME/CFS core groups are used by default.
ME_CFS_CORE_LOWER = [
    "Parkinson's disease",
    "Multiple sclerosis",
    "Inflammatory bowel disease (IBD)",
    "Rheumatoid arthritis",
    "Lupus",
]

ME_CFS_CORE_HIGHER = [
    "Epilepsy",
    "Asthma",
    "Type 1 diabetes",
]

ME_CFS_OPTIONAL = [
    "HIV/AIDS",
    "Breast cancer",
    "Prostate cancer",
    "Migraine",
    "Endometriosis",
    "ALS",
]

LONG_COVID_SET = [
    "Long COVID",
    "Chronic obstructive pulmonary disease (COPD)",
    "Chronic kidney disease",
    "Type 2 diabetes",
    "Stroke",
]


#### 2.2 | Scope switches


In [ ]:
# Core adult ME/CFS comparators are always included.
INCLUDE_OPTIONAL_ME_COMPARATORS = False
INCLUDE_LONG_COVID_SET = False
EXCLUDE_DISEASES = ["Asthma", "Lupus"]

# Add or override diseases here if needed: {"Display name": 'ClinicalTrials.gov query'}
CUSTOM_DISEASES = {}

disease_list = (
    ["ME/CFS"]
    + ME_CFS_CORE_LOWER
    + ME_CFS_CORE_HIGHER
)
if INCLUDE_OPTIONAL_ME_COMPARATORS:
    disease_list += ME_CFS_OPTIONAL
if INCLUDE_LONG_COVID_SET:
    disease_list += LONG_COVID_SET
disease_list += list(CUSTOM_DISEASES)

# Preserve order, remove duplicates and apply any chart exclusions.
disease_list = [
    disease for disease in dict.fromkeys(disease_list)
    if disease not in EXCLUDE_DISEASES
]
print(f"{len(disease_list)} diseases selected:")
print("\n".join(f"  - {disease}" for disease in disease_list))


#### 2.3 | Search definitions


In [ ]:
# query.cond searches the ClinicalTrials.gov condition search area, including submitted
# conditions, condition MeSH terms, titles and keywords. Quoted phrases reduce noise.
CONDITION_QUERIES = {
    "ME/CFS": (
        '"Myalgic Encephalomyelitis" OR "Chronic Fatigue Syndrome" '
        'OR "ME/CFS" OR "Systemic Exertion Intolerance Disease"'
    ),
    "Parkinson's disease": '"Parkinson Disease" OR "Parkinson\'s Disease"',
    "Multiple sclerosis": '"Multiple Sclerosis"',
    "Inflammatory bowel disease (IBD)": (
        '"Inflammatory Bowel Disease" OR "Crohn Disease" '
        'OR "Crohn\'s Disease" OR "Ulcerative Colitis"'
    ),
    "Rheumatoid arthritis": '"Rheumatoid Arthritis"',
    "Lupus": '"Systemic Lupus Erythematosus" OR "Lupus"',
    "HIV/AIDS": (
        '"HIV Infections" OR "Human Immunodeficiency Virus" '
        'OR "Acquired Immunodeficiency Syndrome" OR "HIV/AIDS"'
    ),
    "Breast cancer": '"Breast Cancer" OR "Breast Neoplasms"',
    "Prostate cancer": '"Prostate Cancer" OR "Prostatic Neoplasms"',
    "Migraine": '"Migraine" OR "Migraine Disorders"',
    "Endometriosis": '"Endometriosis"',
    "ALS": '"Amyotrophic Lateral Sclerosis" OR "Motor Neuron Disease"',
    "Epilepsy": '"Epilepsy"',
    "Asthma": '"Asthma"',
    "Type 1 diabetes": (
        '"Type 1 Diabetes" OR "Type 1 Diabetes Mellitus" '
        'OR "Diabetes Mellitus, Type 1"'
    ),
    "Long COVID": (
        '"Long COVID" OR "Post-COVID Conditions" OR "Post-Acute Sequelae of SARS-CoV-2" '
        'OR "Post-Acute COVID-19 Syndrome"'
    ),
    "Chronic obstructive pulmonary disease (COPD)": (
        '"Chronic Obstructive Pulmonary Disease" OR "COPD"'
    ),
    "Chronic kidney disease": '"Chronic Kidney Disease"',
    "Type 2 diabetes": (
        '"Type 2 Diabetes" OR "Type 2 Diabetes Mellitus" '
        'OR "Diabetes Mellitus, Type 2"'
    ),
    "Stroke": '"Stroke" OR "Cerebrovascular Accident"',
}

CONDITION_QUERIES.update(CUSTOM_DISEASES)

missing_queries = [d for d in disease_list if d not in CONDITION_QUERIES]
if missing_queries:
    raise KeyError(f"Missing condition queries for: {missing_queries}")

selected_queries = {d: CONDITION_QUERIES[d] for d in disease_list}
selected_queries


#### 2.4 | Recruitment status groups, colours and outputs


In [ ]:
STATUS_FILTERS = OrderedDict(
    [
        ("Recruiting", "AREA[OverallStatus]RECRUITING"),
        ("Not yet recruiting", "AREA[OverallStatus]NOT_YET_RECRUITING"),
        ("Active, not recruiting", "AREA[OverallStatus]ACTIVE_NOT_RECRUITING"),
        ("Enrolling by invitation", "AREA[OverallStatus]ENROLLING_BY_INVITATION"),
        ("Completed", "AREA[OverallStatus]COMPLETED"),
        (
            "Stopped early",
            "(AREA[OverallStatus]SUSPENDED OR AREA[OverallStatus]TERMINATED "
            "OR AREA[OverallStatus]WITHDRAWN)",
        ),
    ]
)

STATUS_ORDER = [
    "Recruiting",
    "Not yet recruiting",
    "Active, not recruiting",
    "Enrolling by invitation",
    "Completed",
    "Stopped early",
    "Unknown / other",
]

STATUS_COLORS = {
    "Recruiting": "#FFD8A8",              # Fibromyalgia yellow
    "Not yet recruiting": "#B6FDFF",      # POTS cyan
    "Active, not recruiting": "#7BB3F3",  # ME/CFS blue
    "Enrolling by invitation": "#D7B3FF", # Dysautonomia lavender
    "Completed": "#B4D9B5",               # Chronic Lyme green
    "Stopped early": "#E57399",           # Long COVID pink
    "Unknown / other": "#495057",         # Neutral grey
}

STATUS_OUTPUT_CSV = "[2026-08-01-03].csv"
STATUS_OUTPUT_PNG = "[2026-08-01-03].png"


API_BASE = "https://clinicaltrials.gov/api/v2"
USE_CACHE = True


### 🚰 3 | Data Import - ClinicalTrials.gov API v2


#### 3.1 | API helper

The API is queried for counts only (`pageSize=1`, `countTotal=true`), rather than
downloading tens of thousands of full records. Requests use retry/backoff for transient
errors. ClinicalTrials.gov's dataset version timestamp is saved with every result.


In [ ]:
session = requests.Session()
session.headers.update(
    {
        "User-Agent": (
            "CrunchME-clinical-trials-analysis/1.0 "
            "(https://www.crunchme.org/)"
        )
    }
)

def get_json(url, params=None, attempts=4, timeout=45):
    for attempt in range(attempts):
        try:
            response = session.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as exc:
            if attempt == attempts - 1:
                raise
            wait_seconds = 1.5 * (2 ** attempt)
            print(f"  API retry in {wait_seconds:.1f}s: {exc}")
            time.sleep(wait_seconds)

def clinicaltrials_count(condition_query, extra_filter=None):
    advanced_filter = "AREA[StudyType]INTERVENTIONAL"
    if extra_filter:
        advanced_filter += f" AND ({extra_filter})"

    payload = get_json(
        f"{API_BASE}/studies",
        params={
            "format": "json",
            "query.cond": condition_query,
            "filter.advanced": advanced_filter,
            "pageSize": 1,
            "countTotal": "true",
            "fields": "NCTId",
        },
    )
    return int(payload.get("totalCount", 0))

try:
    api_version_info = get_json(f"{API_BASE}/version")
    DATA_TIMESTAMP = api_version_info.get("dataTimestamp", "unknown")
    API_VERSION = api_version_info.get("apiVersion", "unknown")
except Exception as exc:
    print("Could not retrieve API version metadata:", exc)
    DATA_TIMESTAMP = "unknown"
    API_VERSION = "unknown"

print(f"ClinicalTrials.gov API version: {API_VERSION}")
print(f"Dataset timestamp: {DATA_TIMESTAMP}")


#### 3.2 | Fetch counts


In [ ]:
status_cache_signature = hashlib.sha256(
    json.dumps(
        {
            "queries": selected_queries,
            "status_filters": list(STATUS_FILTERS.items()),
            "study_type": "INTERVENTIONAL",
            "data_timestamp": DATA_TIMESTAMP,
        },
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:12]
STATUS_CACHE_FILE = f"clinical_trials_status_cache_{status_cache_signature}.csv"

if USE_CACHE:
    try:
        status_counts_df = pd.read_csv(STATUS_CACHE_FILE)
        expected_status_columns = {
            "Disease", "Status", "Trials", "Condition query",
            "API version", "Data timestamp", "Retrieved at (UTC)",
        }
        if not expected_status_columns.issubset(status_counts_df.columns):
            raise ValueError("Status cache columns do not match this notebook version.")
        print(f"✓ Loaded cached status data from {STATUS_CACHE_FILE}")
    except (FileNotFoundError, ValueError):
        status_counts_df = None
else:
    status_counts_df = None

if status_counts_df is None:
    status_rows = []
    status_retrieved_at = datetime.now(timezone.utc).isoformat(timespec="seconds")

    for disease, condition_query in selected_queries.items():
        print(f"Fetching recruitment statuses for {disease} ...")
        total = clinicaltrials_count(condition_query)
        categorized_total = 0
        disease_status_rows = []

        for status, status_filter in STATUS_FILTERS.items():
            count = clinicaltrials_count(condition_query, status_filter)
            categorized_total += count
            disease_status_rows.append((status, count))
            time.sleep(0.08)

        residual = total - categorized_total
        if residual < 0:
            raise ValueError(
                f"Status counts exceed total for {disease}: "
                f"{categorized_total} > {total}. Check API query logic."
            )

        disease_status_rows.append(("Unknown / other", residual))
        for status, count in disease_status_rows:
            status_rows.append(
                {
                    "Disease": disease,
                    "Status": status,
                    "Trials": count,
                    "Condition query": condition_query,
                    "API version": API_VERSION,
                    "Data timestamp": DATA_TIMESTAMP,
                    "Retrieved at (UTC)": status_retrieved_at,
                }
            )

    status_counts_df = pd.DataFrame(status_rows)
    if USE_CACHE:
        status_counts_df.to_csv(STATUS_CACHE_FILE, index=False)
        print(f"✓ Cached status data to {STATUS_CACHE_FILE}")

status_counts_df.head(10)


### 🥋 4 | Data Wrangling & Checks


#### 4.1 | Order diseases, validate totals and show exact counts


In [ ]:
status_counts_df = status_counts_df[
    status_counts_df["Disease"].isin(disease_list)
].copy()
status_counts_df["Trials"] = status_counts_df["Trials"].astype(int)
status_counts_df["Status"] = pd.Categorical(
    status_counts_df["Status"],
    categories=list(reversed(STATUS_ORDER)),
    ordered=True,
)

status_totals_df = (
    status_counts_df.groupby("Disease", observed=True, as_index=False)["Trials"]
    .sum()
    .rename(columns={"Trials": "Total"})
    .sort_values("Total", ascending=True)
)
status_disease_order = status_totals_df["Disease"].tolist()
status_counts_df["Disease"] = pd.Categorical(
    status_counts_df["Disease"],
    categories=status_disease_order,
    ordered=True,
)
status_counts_df = status_counts_df.sort_values(["Disease", "Status"])

expected_status_rows = len(disease_list) * len(STATUS_ORDER)
assert len(status_counts_df) == expected_status_rows, (
    f"Expected {expected_status_rows} rows, found {len(status_counts_df)}"
)
assert (status_counts_df["Trials"] >= 0).all()
assert set(status_totals_df["Disease"].astype(str)) == set(disease_list)

status_totals_df["Total label"] = status_totals_df["Total"].map(
    lambda value: f"{value:,}"
)
status_table = (
    status_counts_df.pivot_table(
        index="Disease",
        columns="Status",
        values="Trials",
        aggfunc="sum",
        fill_value=0,
        observed=True,
    )
    .reindex(index=list(reversed(status_disease_order)), columns=STATUS_ORDER)
)
status_table["Total"] = status_table.sum(axis=1)
status_table


### 📊 5 | Chart - Registered Interventional Studies by Recruitment Status


In [ ]:
background = "#121421"
white = "#FFFFFF"
muted = "#ADB5BD"

status_max_total = max(int(status_totals_df["Total"].max()), 1)
status_totals_df["Label position"] = (
    status_totals_df["Total"] + status_max_total * 0.012
)
status_chart_height = 8.46  # Preserve the established 3900 x 2538 px export at 300 dpi.

plot_status = (
    ggplot(
        status_counts_df,
        aes(x="Disease", y="Trials", fill="Status"),
    )
    + geom_col(width=0.78)
    + geom_text(
        data=status_totals_df,
        mapping=aes(
            x="Disease",
            y="Label position",
            label="Total label",
        ),
        inherit_aes=False,
        ha="left",
        color=white,
        size=21,
        fontweight="bold",
        family=BRAND_FONT,
    )
    + coord_flip()
    + labs(
        x="",
        y="Number of registered interventional studies",
        fill="Recruitment status",
    )
    + theme_classic()
    + theme(
        figure_size=(13, status_chart_height),
        text=element_text(size=14, family=BRAND_FONT),
        plot_margin_left=0.025,
        plot_margin_right=0.025,
        plot_margin_top=0.008,
        plot_margin_bottom=0.16,
        axis_title_x=element_text(
            size=21,
            weight="bold",
            color=white,
            margin={"t": 14, "units": "pt"},
            family=BRAND_FONT,
        ),
        axis_text_x=element_text(size=20, color=white, family=BRAND_FONT),
        axis_text_y=element_text(
            size=22,
            color=white,
            margin={"r": 10, "units": "pt"},
            family=BRAND_FONT,
        ),
        legend_title=element_text(
            size=22,
            weight="bold",
            color=white,
            ha="left",
            margin={"b": 16, "units": "pt"},
            family=BRAND_FONT,
        ),
        legend_text=element_text(size=20, color=white, family=BRAND_FONT),
        legend_position=(-0.27, -0.34),
        legend_direction="horizontal",
        legend_title_position="top",
        legend_justification="left",
        legend_box_just="left",
        plot_background=element_rect(fill=background, color=background, size=0),
        panel_background=element_rect(fill=background, color=background, size=0),
        legend_background=element_rect(fill=background, color=background, size=0),
        legend_key=element_rect(fill=background, color=background, size=0),
        axis_line=element_line(color=white, size=0.8),
        axis_ticks=element_line(color=white, size=0.8),
        panel_grid_major_x=element_line(color=white, size=0.45, alpha=0.22),
        panel_grid_major_y=element_blank(),
        panel_grid_minor=element_blank(),
    )
    + scale_y_continuous(labels=comma_format(), expand=(0, 0, 0.10, 0))
    + scale_x_discrete(
        labels=lambda labels: [
            "IBD" if label == "Inflammatory bowel disease (IBD)" else label
            for label in labels
        ]
    )
    + scale_fill_manual(values=STATUS_COLORS, breaks=STATUS_ORDER)
    + guides(fill=guide_legend(nrow=2, byrow=True))
)

plot_status


### 💾 6 | Export


In [ ]:
status_export_df = status_counts_df.copy()
status_export_df["Disease"] = status_export_df["Disease"].astype(str)
status_export_df["Status"] = status_export_df["Status"].astype(str)
status_export_df.to_csv(STATUS_OUTPUT_CSV, index=False)
plot_status.save(
    STATUS_OUTPUT_PNG,
    width=13,
    height=status_chart_height,
    dpi=300,
    facecolor=background,
    edgecolor=background,
    verbose=False,
)

print(f"✓ Saved status data:  {STATUS_OUTPUT_CSV}")
print(f"✓ Saved status chart: {STATUS_OUTPUT_PNG}")

# Uncomment in Colab to download both files:
# from google.colab import files
# files.download(STATUS_OUTPUT_CSV)
# files.download(STATUS_OUTPUT_PNG)


### Methodology and interpretation notes

- **Coverage:** This is a comparison of registrations on ClinicalTrials.gov, not a complete
  census of every trial conducted worldwide. Registration rules and historical coverage
  vary by country, sponsor, intervention type and time period.
- **Study type:** Only records classified as `INTERVENTIONAL` are counted. Observational
  studies and expanded-access records are excluded.
- **Recruitment status:** The chart uses the study-level `OverallStatus` field. `Suspended`,
  `Terminated` and `Withdrawn` are grouped as `Stopped early`; residual statuses are retained
  as `Unknown / other`.
- **Condition matching:** Queries use the ClinicalTrials.gov condition search area and
  disease synonyms shown in section 2.3. A study covering multiple selected conditions can
  legitimately appear in more than one disease bar.
- **Reproducibility:** The API version, source-data timestamp, retrieval time and exact
  condition query are included in the exported CSV. Delete the generated cache file to
  force a fresh API pull.
